# Problem Statement

## Business Context

A growing retail bank operating across several Asian markets is facing an increase in customer churn, with customers closing accounts or moving to competitors. This is affecting revenue, customer loyalty, and long-term growth.

The bank wants to use customer data to better understand the reasons behind churn and identify customers who are likely to leave. The key challenges are understanding varied customer behavior and moving from reactive retention efforts to a more proactive, data-driven approach.

## Objective

To overcome the limitations of traditional machine learning workflows, such as manual execution of data preparation, model training, testing, versioning, and deployment, the organization has hired you as a data scientist to implement a robust MLOps pipeline using GitHub Actions on Hugging Face. The objective is to build an automated and reproducible MLOps pipeline that streamlines the entire ML lifecycle - from code integration to model deployment - ensuring faster, more reliable access to the churn prediction model for geographically distributed teams, and enabling proactive, data-driven customer retention strategies.


# Prerequisites

* Create a GitHub repo
    - Go to ***GitHub Profile***
    - Click on ***Your repositories*** then select ***New***
      - Repository Name: ***MLOps***
      - Check the box ***README.md*** file
      - Click on ***Create repository***

* Adding Hugging Face space secrets to GitHub Actions to execute the workflow
  1. Go to Hugging Face ***Profile***
  2. Navigate to ***Access Token***
  3. Create a ***New token***
      - Token type ***Write***
      - Token Name ***MLOps***
      - Click on ***Create Token***
      - Copy the generated Token
  4. Now, go to GitHub repo
      - Click on ***Settings***
      - Navigate to ***Secrets and Variables***
      - Click on ***Actions***
      - Add a ***Repository secerts***
        - Name ***HF_TOKEN***
        - Secret: ***Paste the token created from the hugging face access tokens***
        - Click on ***Add secret***

* Create a Hugging Face space
    - Go to **Hugging Face**
    - Open your **Profile**
    - Click on **New Space**
      - Under the space creation, enter the below details
        - Space name: **Bank-Customer-Churn**
    (If you were trying with different names, be cautious when using a underscore `_` in space names, such as `frontend_space`, as it can cause exceptions when accessing the API URL. Always use an hyphen `-` instead, like `frontend-space`.)
        - Select the space SDK: **Docker**
        - Choose a Docker template: **Streamlit**
        - Click on **Create Space**

In [1]:
pwd

'c:\\Users\\J P Agarwal\\Desktop\\MLOps\\project-3 With MFLOW and CD\\mlops'

In [2]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("mlops", exist_ok=True)

# Model Building

## Data Registration

In [3]:
os.makedirs("mlops/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **bank_customer_churn.csv** in to the folder

In [4]:
# Create a folder for storing the model building files
os.makedirs("mlops/model_building", exist_ok=True)

In [5]:
%%writefile mlops/model_building/data_register.py
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
from huggingface_hub import HfApi, create_repo
import os


repo_id = "krish21may/Bank-Customer-Churn-4"
repo_type = "dataset"

# Initialize API client
api = HfApi(token=os.getenv("HF_TOKEN"))

# Step 1: Check if the space exists
try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists. Using it.")
except RepositoryNotFoundError:
    print(f"Space '{repo_id}' not found. Creating new space...")
    create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
    print(f"Space '{repo_id}' created.")

api.upload_folder(
    folder_path="mlops/data",
    repo_id=repo_id,
    repo_type=repo_type,
)

Writing mlops/model_building/data_register.py


## Data Preparation

In [6]:
%%writefile mlops/model_building/prep.py
# for data manipulation
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# Define constants for the dataset and output paths
api = HfApi(token=os.getenv("HF_TOKEN"))
DATASET_PATH = "hf://datasets/krish21may/Bank-Customer-Churn-4/bank_customer_churn.csv"
bank_dataset = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")

# Define the target variable for the classification task
target = 'Exited'

# List of numerical features in the dataset
numeric_features = [
    'CreditScore',       # Customer's credit score
    'Age',               # Customer's age
    'Tenure',            # Number of years the customer has been with the bank
    'Balance',           # Customer’s account balance
    'NumOfProducts',     # Number of products the customer has with the bank
    'HasCrCard',         # Whether the customer has a credit card (binary: 0 or 1)
    'IsActiveMember',    # Whether the customer is an active member (binary: 0 or 1)
    'EstimatedSalary'    # Customer’s estimated salary
]

# List of categorical features in the dataset
categorical_features = [
    'Geography',         # Country where the customer resides
]

# Define predictor matrix (X) using selected numeric and categorical features
X = bank_dataset[numeric_features + categorical_features]

# Define target variable
y = bank_dataset[target]


# Split dataset into train and test
# Split the dataset into training and test sets
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,              # Predictors (X) and target variable (y)
    test_size=0.2,     # 20% of the data is reserved for testing
    random_state=42    # Ensures reproducibility by setting a fixed random seed
)

Xtrain.to_csv("Xtrain.csv",index=False)
Xtest.to_csv("Xtest.csv",index=False)
ytrain.to_csv("ytrain.csv",index=False)
ytest.to_csv("ytest.csv",index=False)


files = ["Xtrain.csv","Xtest.csv","ytrain.csv","ytest.csv"]

for file_path in files:
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=file_path.split("/")[-1],  # just the filename
        repo_id="krish21may/Bank-Customer-Churn-4",
        repo_type="dataset",
    )

Writing mlops/model_building/prep.py


## Model Training

### Experimentation and Tracking (Development Environment)

In [ ]:
# JP - I ran only the ngrok part in git bash

#!pip install mlflow==3.0.1 pyngrok==7.2.12 -q

To get the ngrok authorization token, please go to this [link](https://dashboard.ngrok.com/authtokens), generate a new token, copy it, and paste it in the designated code line below.

In [10]:
import subprocess
import sys
import time
import webbrowser

# Start MLflow Server
process = subprocess.Popen([
    sys.executable,
    "-m",
    "mlflow",
    "server",
    "--host",
    "127.0.0.1",
    "--port",
    "5000"
])

# Wait for the server to start
time.sleep(5)

# MLflow Local URL
mlflow_url = "http://127.0.0.1:5000"

print("=" * 60)
print("MLflow UI is available at:")
print(mlflow_url)
print("=" * 60)

# Open automatically in the default browser
webbrowser.open(mlflow_url)

MLflow UI is available at:
http://127.0.0.1:5000


True

In [12]:
# Set the tracking URL for MLflow
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("MLOps_experiment10")

2026/07/12 03:09:33 INFO mlflow.tracking.fluent: Experiment with name 'MLOps_experiment10' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/905550251645901631', creation_time=1783805973118, experiment_id='905550251645901631', last_update_time=1783805973118, lifecycle_stage='active', name='MLOps_experiment10', tags={}>

In [14]:
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, recall_score
# for model serialization
import joblib


bank_dataset = pd.read_csv("mlops/data/bank_customer_churn.csv")
print("Dataset loaded successfully.")

# Define the target variable for the classification task
target = 'Exited'

# List of numerical features in the dataset
numeric_features = [
    'CreditScore',       # Customer's credit score
    'Age',               # Customer's age
    'Tenure',            # Number of years the customer has been with the bank
    'Balance',           # Customer’s account balance
    'NumOfProducts',     # Number of products the customer has with the bank
    'HasCrCard',         # Whether the customer has a credit card (binary: 0 or 1)
    'IsActiveMember',    # Whether the customer is an active member (binary: 0 or 1)
    'EstimatedSalary'    # Customer’s estimated salary
]

# List of categorical features in the dataset
categorical_features = [
    'Geography',         # Country where the customer resides
]

# Define predictor matrix (X) using selected numeric and categorical features
X = bank_dataset[numeric_features + categorical_features]

# Define target variable
y = bank_dataset[target]


# Split dataset into train and test
# Split the dataset into training and test sets
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,              # Predictors (X) and target variable (y)
    test_size=0.2,     # 20% of the data is reserved for testing
    random_state=42    # Ensures reproducibility by setting a fixed random seed
)

# Set the clas weight to handle class imbalance
class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]
class_weight

# Define the preprocessing steps
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Define base XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=class_weight, random_state=42)


# Define hyperparameter grid
param_grid = {
    'xgbclassifier__n_estimators': [50, 75, 100],    # number of tree to build
    'xgbclassifier__max_depth': [2, 3],    # maximum depth of each tree
    'xgbclassifier__colsample_bytree': [0.4, 0.6],    # percentage of attributes to be considered (randomly) for each tree
    'xgbclassifier__colsample_bylevel': [0.4, 0.6],    # percentage of attributes to be considered (randomly) for each level of a tree
    'xgbclassifier__learning_rate': [0.01, 0.1],    # learning rate
    'xgbclassifier__reg_lambda': [0.4, 0.6],    # L2 regularization factor
}

# Model pipeline
model_pipeline = make_pipeline(preprocessor, xgb_model)

with mlflow.start_run():
    # Hyperparameter tuning
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, n_jobs=-1)
    grid_search.fit(Xtrain, ytrain)

    # Log all parameter combinations and their mean test scores
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]
        std_score = results['std_test_score'][i]

        # Log each combination as a separate MLflow run
        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_test_score", mean_score)
            mlflow.log_metric("std_test_score", std_score)

    # Log best parameters separately in main run
    mlflow.log_params(grid_search.best_params_)

    # Store and evaluate the best model
    best_model = grid_search.best_estimator_

    classification_threshold = 0.45

    y_pred_train_proba = best_model.predict_proba(Xtrain)[:, 1]
    y_pred_train = (y_pred_train_proba >= classification_threshold).astype(int)

    y_pred_test_proba = best_model.predict_proba(Xtest)[:, 1]
    y_pred_test = (y_pred_test_proba >= classification_threshold).astype(int)

    train_report = classification_report(ytrain, y_pred_train, output_dict=True)
    test_report = classification_report(ytest, y_pred_test, output_dict=True)

    mlflow.log_metrics({
        "train_accuracy": train_report['accuracy'],
        "train_precision": train_report['1']['precision'],
        "train_recall": train_report['1']['recall'],
        "train_f1-score": train_report['1']['f1-score'],
        "test_accuracy": test_report['accuracy'],
        "test_precision": test_report['1']['precision'],
        "test_recall": test_report['1']['recall'],
        "test_f1-score": test_report['1']['f1-score']
    })

Dataset loaded successfully.
🏃 View run invincible-toad-931 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/e3f365fd93ee473fa54fce80fc5296df
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run flawless-ant-217 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/9fcb626eacf8499e981f19bb7be87b9d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run thoughtful-ox-305 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/7fc096ec35db4a85885ea6ec6c0be137
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run debonair-tern-795 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/14eb2469ca0045fc89e720ee261d40bb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/905550251645901631
🏃 View run dazzling-shrike-351 at: http://127.0.0.1:5000/#/experiments/905550251645901631/runs/189e7ccbe1db422cac0e4f6a5b735ce4
🧪 View experiment at: http://1

- As we can see, all the experiments conducted during hyperparameter tuning are being logged by MLflow.
- Upon clicking the links in the output of the above code cell, we can check the tracking on MLflow.

Now that we've tested the experimentation tracking with MLflow in a development environment, let's convert this to the required script for production environment usage.

### Experimentation and Tracking (Production Environment)

In [15]:
%%writefile mlops/model_building/train.py
# for data manipulation
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, recall_score
# for model serialization
import joblib
# for creating a folder
import os
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mlops-training-experiment")

api = HfApi()


Xtrain_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/Xtrain.csv"
Xtest_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/Xtest.csv"
ytrain_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/ytrain.csv"
ytest_path = "hf://datasets/krish21may/Bank-Customer-Churn-4/ytest.csv"

Xtrain = pd.read_csv(Xtrain_path)
Xtest = pd.read_csv(Xtest_path)
ytrain = pd.read_csv(ytrain_path)
ytest = pd.read_csv(ytest_path)


# List of numerical features in the dataset
numeric_features = [
    'CreditScore',       # Customer's credit score
    'Age',               # Customer's age
    'Tenure',            # Number of years the customer has been with the bank
    'Balance',           # Customer’s account balance
    'NumOfProducts',     # Number of products the customer has with the bank
    'HasCrCard',         # Whether the customer has a credit card (binary: 0 or 1)
    'IsActiveMember',    # Whether the customer is an active member (binary: 0 or 1)
    'EstimatedSalary'    # Customer’s estimated salary
]

# List of categorical features in the dataset
categorical_features = [
    'Geography',         # Country where the customer resides
]


# Set the clas weight to handle class imbalance
class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]
class_weight

# Define the preprocessing steps
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Define base XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=class_weight, random_state=42)

# Define hyperparameter grid
param_grid = {
    'xgbclassifier__n_estimators': [50, 75, 100, 125, 150],    # number of tree to build
    'xgbclassifier__max_depth': [2, 3, 4],    # maximum depth of each tree
    'xgbclassifier__colsample_bytree': [0.4, 0.5, 0.6],    # percentage of attributes to be considered (randomly) for each tree
    'xgbclassifier__colsample_bylevel': [0.4, 0.5, 0.6],    # percentage of attributes to be considered (randomly) for each level of a tree
    'xgbclassifier__learning_rate': [0.01, 0.05, 0.1],    # learning rate
    'xgbclassifier__reg_lambda': [0.4, 0.5, 0.6],    # L2 regularization factor
}

# Model pipeline
model_pipeline = make_pipeline(preprocessor, xgb_model)

# Start MLflow run
with mlflow.start_run():
    # Hyperparameter tuning
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, n_jobs=-1)
    grid_search.fit(Xtrain, ytrain)

    # Log all parameter combinations and their mean test scores
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]
        std_score = results['std_test_score'][i]

        # Log each combination as a separate MLflow run
        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_test_score", mean_score)
            mlflow.log_metric("std_test_score", std_score)

    # Log best parameters separately in main run
    mlflow.log_params(grid_search.best_params_)

    # Store and evaluate the best model
    best_model = grid_search.best_estimator_

    classification_threshold = 0.45

    y_pred_train_proba = best_model.predict_proba(Xtrain)[:, 1]
    y_pred_train = (y_pred_train_proba >= classification_threshold).astype(int)

    y_pred_test_proba = best_model.predict_proba(Xtest)[:, 1]
    y_pred_test = (y_pred_test_proba >= classification_threshold).astype(int)

    train_report = classification_report(ytrain, y_pred_train, output_dict=True)
    test_report = classification_report(ytest, y_pred_test, output_dict=True)

    # Log the metrics for the best model
    mlflow.log_metrics({
        "train_accuracy": train_report['accuracy'],
        "train_precision": train_report['1']['precision'],
        "train_recall": train_report['1']['recall'],
        "train_f1-score": train_report['1']['f1-score'],
        "test_accuracy": test_report['accuracy'],
        "test_precision": test_report['1']['precision'],
        "test_recall": test_report['1']['recall'],
        "test_f1-score": test_report['1']['f1-score']
    })

    # Save the model locally
    model_path = "best_churn_model_v1.joblib"
    joblib.dump(best_model, model_path)

    # Log the model artifact
    mlflow.log_artifact(model_path, artifact_path="model")
    print(f"Model saved as artifact at: {model_path}")

    # Upload to Hugging Face
    repo_id = "krish21may/Bank-Customer-Churn-4"
    repo_type = "model"

    # Step 1: Check if the space exists
    try:
        api.repo_info(repo_id=repo_id, repo_type=repo_type)
        print(f"Space '{repo_id}' already exists. Using it.")
    except RepositoryNotFoundError:
        print(f"Space '{repo_id}' not found. Creating new space...")
        create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
        print(f"Space '{repo_id}' created.")

    # create_repo("churn-model", repo_type="model", private=False)
    api.upload_file(
        path_or_fileobj="best_churn_model_v1.joblib",
        path_in_repo="best_churn_model_v1.joblib",
        repo_id=repo_id,
        repo_type=repo_type,
    )

Writing mlops/model_building/train.py


# Deployment

## Dockerfile

In [16]:
os.makedirs("mlops/deployment", exist_ok=True)

In [17]:
%%writefile mlops/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing mlops/deployment/Dockerfile


## Streamlit App

In [23]:
%%writefile mlops/deployment/app.py
import streamlit as st
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
import joblib
import io
import sys
import os

# Ensure current working directory & parents are in module path
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath(".."))

import plotly.express as px
import plotly.graph_objects as go

try:
    from mlops.analytics.shap_explainer import calculate_shap_contributions
    from mlops.analytics.roi_calculator import calculate_clv, calculate_expected_retention_roi, optimize_decision_threshold
    from mlops.analytics.counterfactual import generate_counterfactual_scenarios
    from mlops.analytics.survival_analysis import predict_survival_timeline
    from mlops.analytics.uplift_modeling import segment_causal_uplift
    from mlops.analytics.llm_outreach import generate_llm_retention_outreach
    from mlops.analytics.fairness_audit import run_fairness_audit
    from mlops.analytics.monte_carlo_sim import run_monte_carlo_simulation
    from mlops.monitoring.drift_monitor import run_drift_analysis
    from mlops.reports.pdf_generator import generate_executive_pdf_report
except ModuleNotFoundError:
    try:
        from analytics.shap_explainer import calculate_shap_contributions
        from analytics.roi_calculator import calculate_clv, calculate_expected_retention_roi, optimize_decision_threshold
        from analytics.counterfactual import generate_counterfactual_scenarios
        from analytics.survival_analysis import predict_survival_timeline
        from analytics.uplift_modeling import segment_causal_uplift
        from analytics.llm_outreach import generate_llm_retention_outreach
        from analytics.fairness_audit import run_fairness_audit
        from analytics.monte_carlo_sim import run_monte_carlo_simulation
        from monitoring.drift_monitor import run_drift_analysis
        from reports.pdf_generator import generate_executive_pdf_report
    except Exception as import_err:
        st.error(f"Module import notice: {import_err}")

# Set Page Config
st.set_page_config(
    page_title="Bank Customer Churn Intelligence Platform",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Sidebar Controls & Theme Selector
with st.sidebar:
    st.image("https://img.icons8.com/isometric/100/bank.png", width=70)
    st.title("Control Panel")
    
    st.markdown("### 🎨 Visual Theme")
    theme_mode = st.radio("Select Theme:", ["🌙 Dark Mode", "☀️ Light Mode"], index=0)
    is_dark = "Dark" in theme_mode

    st.markdown("---")
    st.markdown("### ⚙️ System Architecture")
    st.info("""
    - **Model**: Tuned XGBoost Classifier
    - **XAI**: SHAP Local Force Attribution
    - **Recourse**: DiCE Counterfactual AI
    - **Survival**: Cox Hazard 24M Curves
    - **Causal**: Uplift Segmentation
    - **Fairness**: 4/5th Rule ECOA Audit
    """)

# Dynamic Theme Tokens
if is_dark:
    bg_color, card_bg, text_primary, text_secondary = "#0F172A", "#1E293B", "#F8FAFC", "#94A3B8"
    border_color, accent_color, plotly_template = "#334155", "#38BDF8", "plotly_dark"
    card_shadow = "0 4px 12px rgba(0, 0, 0, 0.4)"
else:
    bg_color, card_bg, text_primary, text_secondary = "#F8FAFC", "#FFFFFF", "#0F172A", "#475569"
    border_color, accent_color, plotly_template = "#E2E8F0", "#2563EB", "plotly_white"
    card_shadow = "0 4px 6px -1px rgba(0, 0, 0, 0.05)"

st.markdown(f"""
<style>
    .stApp {{ background-color: {bg_color}; color: {text_primary}; }}
    .main-header {{ font-size: 2.2rem; font-weight: 800; color: {accent_color}; margin-bottom: 0.2rem; }}
    .sub-header {{ font-size: 1.05rem; color: {text_secondary}; margin-bottom: 1.5rem; }}
    .stButton>button {{
        background-color: {accent_color}; color: #FFFFFF !important; font-size: 1.05rem; font-weight: 600;
        border-radius: 8px; padding: 0.65rem 2rem; width: 100%; border: none; transition: all 0.3s ease;
    }}
    .stButton>button:hover {{ opacity: 0.9; transform: translateY(-1px); }}
</style>
""", unsafe_allow_html=True)

# Load Model
@st.cache_resource
def load_churn_model():
    try:
        model_path = hf_hub_download(repo_id="krish21may/Bank-Customer-Churn-4", filename="best_churn_model.joblib")
        return joblib.load(model_path)
    except Exception as e:
        st.error(f"Error loading model from Hugging Face Hub: {e}")
        return None

model = load_churn_model()

# Header Banner
st.markdown('<div class="main-header">🏦 Bank Customer Churn Intelligence & MLOps Platform</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">Enterprise Decision Platform: SHAP XAI • Counterfactual Recourse • Survival Curves • Causal Uplift • Executive PDF</div>', unsafe_allow_html=True)

# Navigation Tabs
tab_single, tab_survival, tab_causal, tab_batch, tab_analytics, tab_drift, tab_pdf = st.tabs([
    "👤 Single Risk & SHAP XAI", 
    "⏳ Survival & Timeline",
    "🎯 Causal Uplift Matrix",
    "📁 Batch CSV Processor", 
    "📊 Portfolio Analytics",
    "⚖️ Fair Lending & Drift",
    "📄 Executive PDF Briefing"
])

REQUIRED_COLUMNS = ['CreditScore', 'Geography', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

# ==============================================================================
# TAB 1: SINGLE CUSTOMER PREDICTION & SHAP & COUNTERFACTUAL & LLM OUTREACH
# ==============================================================================
with tab_single:
    st.markdown("### 👤 Single Customer Analysis & SHAP Explainability")
    
    preset = st.radio("Load Scenario Profile:", ["Custom Input", "⚠️ High Churn Risk Profile", "✅ Low Churn Risk Profile"], horizontal=True)
    
    if preset == "⚠️ High Churn Risk Profile":
        def_credit, def_geo, def_age, def_tenure, def_balance, def_num_prod, def_card, def_active, def_salary = 590, "Germany", 52, 2, 125000.0, 1, "Yes", "No", 75000.0
    elif preset == "✅ Low Churn Risk Profile":
        def_credit, def_geo, def_age, def_tenure, def_balance, def_num_prod, def_card, def_active, def_salary = 750, "France", 28, 7, 45000.0, 2, "Yes", "Yes", 95000.0
    else:
        def_credit, def_geo, def_age, def_tenure, def_balance, def_num_prod, def_card, def_active, def_salary = 650, "France", 38, 5, 50000.0, 1, "Yes", "Yes", 60000.0

    col_input, col_results = st.columns([1.1, 1], gap="large")

    with col_input:
        st.markdown("##### 📋 Demographic & Financial Inputs")
        c1, c2 = st.columns(2)
        with c1:
            Age = st.number_input("Age (Years)", min_value=18, max_value=100, value=def_age, step=1, key="s_age")
            Geography = st.selectbox("Geography", ["France", "Germany", "Spain"], index=["France", "Germany", "Spain"].index(def_geo), key="s_geo")
            Tenure = st.number_input("Tenure (Years)", min_value=0, max_value=20, value=def_tenure, step=1, key="s_tenure")
            EstimatedSalary = st.number_input("Estimated Salary ($)", min_value=0.0, max_value=500000.0, value=float(def_salary), step=1000.0, key="s_salary")
        with c2:
            CreditScore = st.number_input("Credit Score", min_value=300, max_value=900, value=def_credit, step=5, key="s_credit")
            Balance = st.number_input("Account Balance ($)", min_value=0.0, max_value=1000000.0, value=float(def_balance), step=5000.0, key="s_balance")
            NumOfProducts = st.slider("Number of Products", min_value=1, max_value=4, value=def_num_prod, key="s_products")
            HasCrCard = st.selectbox("Has Credit Card?", ["Yes", "No"], index=0 if def_card == "Yes" else 1, key="s_card")
            IsActiveMember = st.selectbox("Is Active Member?", ["Yes", "No"], index=0 if def_active == "Yes" else 1, key="s_active")

        predict_btn = st.button("🔍 Run Churn Risk & SHAP Analysis")

    single_input = pd.DataFrame([{
        'CreditScore': CreditScore, 'Geography': Geography, 'Age': Age, 'Tenure': Tenure,
        'Balance': Balance, 'NumOfProducts': NumOfProducts,
        'HasCrCard': 1 if HasCrCard == "Yes" else 0,
        'IsActiveMember': 1 if IsActiveMember == "Yes" else 0,
        'EstimatedSalary': EstimatedSalary
    }])

    with col_results:
        st.markdown("##### 📊 Prediction & SHAP Risk Attribution")
        
        if predict_btn or preset != "Custom Input":
            if model is not None:
                prob = float(model.predict_proba(single_input)[0, 1])
                threshold = 0.45
                is_churn = prob >= threshold
                
                # Store in session state for tabs 2 & 3
                st.session_state['single_prob'] = prob
                st.session_state['single_input'] = single_input
                
                # Gauge Chart
                fig_gauge = go.Figure(go.Indicator(
                    mode="gauge+number", value=prob * 100,
                    number={'suffix': '%', 'font': {'size': 34, 'color': accent_color}},
                    title={'text': "Churn Risk Score", 'font': {'size': 18, 'color': text_primary}},
                    gauge={
                        'axis': {'range': [0, 100]},
                        'bar': {'color': "#EF4444" if is_churn else "#10B981"},
                        'bgcolor': card_bg,
                        'steps': [
                            {'range': [0, 30], 'color': 'rgba(16, 185, 129, 0.2)'},
                            {'range': [30, 45], 'color': 'rgba(245, 158, 11, 0.2)'},
                            {'range': [45, 100], 'color': 'rgba(239, 68, 68, 0.2)'}
                        ],
                        'threshold': {'line': {'color': "red", 'width': 3}, 'value': threshold * 100}
                    }
                ))
                fig_gauge.update_layout(template=plotly_template, height=210, margin=dict(l=20, r=20, t=30, b=20), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
                st.plotly_chart(fig_gauge, use_container_width=True)
                
                # SHAP Bar Chart
                shap_df = calculate_shap_contributions(model, single_input)
                fig_shap = px.bar(
                    shap_df, x='SHAP_Impact', y='Feature', orientation='h', color='Impact_Type',
                    color_discrete_map={'Increases Churn Risk ⚠️': '#EF4444', 'Decreases Churn Risk ✅': '#10B981'},
                    title="SHAP Feature Force Drivers", template=plotly_template
                )
                fig_shap.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', height=250)
                st.plotly_chart(fig_shap, use_container_width=True)
                
                # Counterfactual What-If Recourse Scenarios
                st.markdown("##### 💡 Counterfactual 'What-If' Actionable Recourse")
                cf_scenarios = generate_counterfactual_scenarios(model, single_input)
                for sc in cf_scenarios:
                    with st.expander(f"{sc['Scenario_Name']} (Risk Drops: {sc['Original_Risk_%']}% ➔ {sc['New_Risk_%']}%)"):
                        for act in sc['Actions_Required']:
                            st.write(f"- {act}")

                # LLM Outreach
                st.markdown("##### ✉️ LLM Generated Personalized Retention Copy")
                outreach = generate_llm_retention_outreach(single_input.iloc[0].to_dict(), prob, [])
                with st.expander("📄 View Generated Customer Retention Email & SMS"):
                    st.code(outreach['Email_Body'], language="markdown")
                    st.caption(f"SMS Copy: {outreach['SMS_Copy']}")
            else:
                st.warning("Model not loaded.")
        else:
            st.info("👈 Set customer details on the left and click **'Run Churn Risk & SHAP Analysis'**.")

# ==============================================================================
# TAB 2: SURVIVAL ANALYSIS & TIME-TO-CHURN
# ==============================================================================
with tab_survival:
    st.markdown("### ⏳ Survival Analysis & Time-to-Churn Timeline")
    st.markdown("Models 24-month customer retention curves and predicts expected customer lifespan before attrition.")
    
    current_prob = st.session_state.get('single_prob', 0.65)
    surv_info = predict_survival_timeline(current_prob)
    
    s1, s2, s3 = st.columns(3)
    s1.metric("Est. Customer Lifespan", f"{surv_info['Expected_Months_Until_Churn']} Months")
    s2.metric("6-Month Survival Rate", f"{surv_info['Prob_Survival_6M_%']}%")
    s3.metric("Hazard Risk Category", surv_info['Hazard_Risk_Category'])
    
    st.markdown("##### 📈 24-Month Customer Survival Probability Curve")
    fig_surv = px.line(
        surv_info['Survival_Curve_DF'], x='Month', y='Survival_Probability_%',
        title="24-Month Retention Probability Trajectory", markers=True, template=plotly_template
    )
    fig_surv.update_traces(line_color=accent_color, line_width=3)
    fig_surv.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', height=350)
    st.plotly_chart(fig_surv, use_container_width=True)

# ==============================================================================
# TAB 3: CAUSAL ML & UPLIFT MATRIX
# ==============================================================================
with tab_causal:
    st.markdown("### 🎯 Causal ML & Uplift Campaign Matrix")
    st.markdown("Identifies **Persuadables** (customers who stay *only* if offered a retention incentive) to maximize marketing ROI.")
    
    sample_uplift_df = pd.DataFrame([
        {'CreditScore': 619, 'Geography': 'France', 'Age': 42, 'Balance': 85000.0, 'NumOfProducts': 1, 'IsActiveMember': 0, 'Churn_Probability': 0.62},
        {'CreditScore': 608, 'Geography': 'Spain', 'Age': 31, 'Balance': 45000.0, 'NumOfProducts': 2, 'IsActiveMember': 1, 'Churn_Probability': 0.18},
        {'CreditScore': 502, 'Geography': 'Germany', 'Age': 58, 'Balance': 150000.0, 'NumOfProducts': 3, 'IsActiveMember': 0, 'Churn_Probability': 0.88},
        {'CreditScore': 699, 'Geography': 'France', 'Age': 39, 'Balance': 0.0, 'NumOfProducts': 1, 'IsActiveMember': 0, 'Churn_Probability': 0.42}
    ])
    
    res_uplift = segment_causal_uplift(sample_uplift_df)
    
    u1, u2 = st.columns(2)
    with u1:
        st.markdown("##### 🎯 Causal Segment Breakdown")
        fig_causal = px.pie(res_uplift, names='Causal_Segment', title="Portfolio Uplift Segments", hole=0.4, template=plotly_template)
        fig_causal.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_causal, use_container_width=True)
        
    with u2:
        st.markdown("##### 💡 Campaign Targeting Guidelines")
        st.info("""
        - 🎯 **Persuadables**: High Uplift — Allocate 80% of retention campaign budget here.
        - 🔒 **Sure Things**: Low Churn Risk — Do not spend retention budget.
        - ❌ **Lost Causes**: Extremely high risk / inactive — Low campaign response.
        - ⚠️ **Sleeping Dogs**: Low risk — Do not disturb with unneeded emails.
        """)

# ==============================================================================
# TAB 4: BATCH CSV PROCESSOR
# ==============================================================================
with tab_batch:
    st.markdown("### 📁 Batch Prediction Engine & CLV Optimizer")
    
    c_up, c_template = st.columns([2, 1], gap="large")
    with c_template:
        sample_df = pd.DataFrame([
            {'CreditScore': 619, 'Geography': 'France', 'Age': 42, 'Tenure': 2, 'Balance': 0.0, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 101348.88},
            {'CreditScore': 608, 'Geography': 'Spain', 'Age': 41, 'Tenure': 1, 'Balance': 83807.86, 'NumOfProducts': 1, 'HasCrCard': 0, 'IsActiveMember': 1, 'EstimatedSalary': 112542.58}
        ])
        buf = io.BytesIO()
        sample_df.to_csv(buf, index=False)
        st.download_button("📄 Download Sample CSV Template", buf.getvalue(), "sample_bank_customers.csv", "text/csv")
        
    with c_up:
        uploaded_file = st.file_uploader("Upload Customer CSV File", type=["csv"], key="b_uploader")

    if uploaded_file is not None:
        try:
            batch_df = pd.read_csv(uploaded_file)
            st.success(f"File **'{uploaded_file.name}'** loaded ({len(batch_df)} records).")
            
            COLUMN_MAP = {
                'creditscore': 'CreditScore', 'credit_score': 'CreditScore', 'score': 'CreditScore',
                'geography': 'Geography', 'country': 'Geography', 'location': 'Geography',
                'age': 'Age',
                'tenure': 'Tenure', 'years': 'Tenure',
                'balance': 'Balance', 'account_balance': 'Balance',
                'numofproducts': 'NumOfProducts', 'num_of_products': 'NumOfProducts', 'products': 'NumOfProducts',
                'hascrcard': 'HasCrCard', 'has_cr_card': 'HasCrCard', 'credit_card': 'HasCrCard',
                'isactivemember': 'IsActiveMember', 'is_active_member': 'IsActiveMember', 'active': 'IsActiveMember',
                'estimatedsalary': 'EstimatedSalary', 'estimated_salary': 'EstimatedSalary', 'salary': 'EstimatedSalary'
            }

            DEFAULT_VALUES = {
                'CreditScore': 650, 'Geography': 'France', 'Age': 38, 'Tenure': 5,
                'Balance': 50000.0, 'NumOfProducts': 1, 'HasCrCard': 1, 'IsActiveMember': 1,
                'EstimatedSalary': 75000.0
            }

            # Map column names
            renamed_cols = {}
            for col in batch_df.columns:
                clean_col = str(col).strip().lower().replace(" ", "_")
                if clean_col in COLUMN_MAP:
                    renamed_cols[col] = COLUMN_MAP[clean_col]

            mapped_df = batch_df.rename(columns=renamed_cols).copy()

            # Identify missing required bank columns
            missing_cols = [c for c in REQUIRED_COLUMNS if c not in mapped_df.columns]
            if missing_cols:
                st.info(f"💡 Notice: File '{uploaded_file.name}' did not contain standard banking column headers. Missing fields ({missing_cols}) were auto-filled with baseline defaults to complete churn risk inference.")
                for m_col in missing_cols:
                    mapped_df[m_col] = DEFAULT_VALUES[m_col]

            proc_df = mapped_df[REQUIRED_COLUMNS].copy()

            # Ensure numeric data types & handle boolean/string values
            for b_col in ['HasCrCard', 'IsActiveMember']:
                proc_df[b_col] = proc_df[b_col].apply(lambda x: 1 if str(x).strip().lower() in ['1', 'yes', 'true'] else (0 if str(x).strip().lower() in ['0', 'no', 'false'] else 1))

            for num_col in ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']:
                proc_df[num_col] = pd.to_numeric(proc_df[num_col], errors='coerce').fillna(DEFAULT_VALUES[num_col])

            proc_df['Geography'] = proc_df['Geography'].astype(str).apply(lambda g: g if g in ['France', 'Germany', 'Spain'] else 'France')

            if model is not None:
                probs = model.predict_proba(proc_df)[:, 1]
                threshold = 0.45
                preds = (probs >= threshold).astype(int)
                
                res_df = batch_df.copy()
                res_df['Churn_Probability'] = probs
                res_df['Churn_Probability_%'] = np.round(probs * 100, 2)
                res_df['Risk_Status'] = np.where(preds == 1, 'HIGH RISK ⚠️', 'LOW RISK ✅')
                
                clvs = [calculate_clv(r['Balance'], r['EstimatedSalary'], r['NumOfProducts'], r['Tenure']) for _, r in proc_df.iterrows()]
                res_df['CLV_$'] = np.round(clvs, 2)
                
                st.session_state['batch_results'] = res_df
                st.session_state['batch_proc_df'] = proc_df
                
                opt_res = optimize_decision_threshold(res_df)
                
                k1, k2, k3, k4 = st.columns(4)
                k1.metric("Total Records", len(res_df))
                k2.metric("High Risk Count", int(np.sum(preds == 1)), f"{np.mean(preds)*100:.1f}% risk rate", delta_color="inverse")
                k3.metric("Optimal Risk Threshold", f"{opt_res['Optimal_Threshold']}")
                k4.metric("Max Projected Net Profit", f"${opt_res['Max_Net_Profit']:,.2f}")

                st.dataframe(res_df, use_container_width=True)
                
                out_buf = io.BytesIO()
                res_df.to_csv(out_buf, index=False)
                st.download_button("📥 Download Batch Predictions CSV", out_buf.getvalue(), f"churn_predictions_{uploaded_file.name}", "text/csv")
        except Exception as err:
            st.error(f"Error parsing CSV: {err}")

# ==============================================================================
# TAB 5: PORTFOLIO VISUAL ANALYTICS
# ==============================================================================
with tab_analytics:
    st.markdown("### 📊 Portfolio Visual Analytics Dashboard")
    
    if 'batch_results' in st.session_state and st.session_state['batch_results'] is not None:
        df_ana = st.session_state['batch_results']
    else:
        df_ana = pd.DataFrame([
            {'CreditScore': 619, 'Geography': 'France', 'Age': 42, 'Tenure': 2, 'Balance': 0.0, 'NumOfProducts': 1, 'Churn_Probability_%': 62.4, 'Risk_Status': 'HIGH RISK ⚠️'},
            {'CreditScore': 608, 'Geography': 'Spain', 'Age': 41, 'Tenure': 1, 'Balance': 83807.86, 'NumOfProducts': 1, 'Churn_Probability_%': 18.2, 'Risk_Status': 'LOW RISK ✅'},
            {'CreditScore': 502, 'Geography': 'Germany', 'Age': 58, 'Tenure': 8, 'Balance': 159660.8, 'NumOfProducts': 3, 'Churn_Probability_%': 74.8, 'Risk_Status': 'HIGH RISK ⚠️'}
        ])

    ca1, ca2 = st.columns(2)
    with ca1:
        st.markdown("##### 📈 Demographics & Account Balance Cluster")
        fig_scatter = px.scatter(df_ana, x='Age', y='Balance', size='Churn_Probability_%' if 'Churn_Probability_%' in df_ana.columns else None, color='Risk_Status' if 'Risk_Status' in df_ana.columns else 'Geography', template=plotly_template)
        fig_scatter.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_scatter, use_container_width=True)

    with ca2:
        st.markdown("##### 🎲 Monte Carlo Portfolio Attrition Simulation")
        mc_res = run_monte_carlo_simulation(df_ana)
        st.metric("95% Value-at-Risk (VaR) Deposit Loss", f"${mc_res['VaR_95_USD']:,.2f}")
        fig_hist = px.histogram(mc_res['Loss_Distribution'], nbins=30, title="Monte Carlo 1,000-Trial Attrition Distribution", template=plotly_template)
        fig_hist.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_hist, use_container_width=True)

# ==============================================================================
# TAB 6: FAIR LENDING & EVIDENTLY DRIFT AUDIT
# ==============================================================================
with tab_drift:
    st.markdown("### ⚖️ Fair Lending Audit & Evidently Data Drift")
    
    if 'batch_proc_df' in st.session_state:
        drift_df = st.session_state['batch_proc_df']
    else:
        drift_df = pd.DataFrame([
            {'CreditScore': 550, 'Geography': 'Germany', 'Age': 60, 'Tenure': 1, 'Balance': 180000.0, 'NumOfProducts': 3, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 120000.0},
            {'CreditScore': 510, 'Geography': 'Spain', 'Age': 58, 'Tenure': 2, 'Balance': 195000.0, 'NumOfProducts': 4, 'HasCrCard': 0, 'IsActiveMember': 0, 'EstimatedSalary': 130000.0}
        ])
        
    f_res = run_fairness_audit(drift_df)
    d_res = run_drift_analysis(drift_df)
    
    st.markdown("##### ⚖️ ECOA Fair Lending Disparate Impact Audit")
    st.metric("Disparate Impact Ratio (4/5th Rule)", f"{f_res['Disparate_Impact_Ratio']}", f_res['Regulatory_Status'])
    
    st.markdown("##### 📉 Evidently AI KS-Test Data Drift Status")
    st.metric("Overall Drift Status", "DRIFT DETECTED ⚠️" if d_res['Drift_Detected'] else "HEALTHY (NO DRIFT) ✅", f"Drifted Share: {d_res['Drift_Share_%']}%")

# ==============================================================================
# TAB 7: EXECUTIVE PDF BRIEFING & FASTAPI DOCS
# ==============================================================================
with tab_pdf:
    st.markdown("### 📄 Download C-Suite Executive PDF Briefing")
    
    if 'batch_results' in st.session_state and st.session_state['batch_results'] is not None:
        pdf_df = st.session_state['batch_results']
    else:
        pdf_df = pd.DataFrame([
            {'CreditScore': 619, 'Geography': 'France', 'Age': 42, 'Balance': 85000.0, 'Churn_Probability_%': 62.4, 'Risk_Status': 'HIGH RISK ⚠️'},
            {'CreditScore': 608, 'Geography': 'Spain', 'Age': 41, 'Balance': 45000.0, 'Churn_Probability_%': 18.2, 'Risk_Status': 'LOW RISK ✅'}
        ])
        
    pdf_bytes = generate_executive_pdf_report(pdf_df)
    
    st.download_button(
        "📄 Download Publication-Ready Executive PDF Briefing",
        pdf_bytes,
        "executive_churn_intelligence_briefing.pdf",
        "application/pdf"
    )
    
    st.markdown("---")
    st.markdown("##### ⚡ FastAPI REST Microservice Documentation")
    st.code("uvicorn mlops.api.main:app --host 0.0.0.0 --port 8000 --reload", language="bash")
    st.info("Interactive OpenAPI / Swagger docs available at `http://localhost:8000/docs` when running.")

# Footer
st.markdown("---")
st.markdown("<div style='text-align: center; color: #9CA3AF;'>Bank Customer Churn Intelligence Platform | XGBoost • SHAP • DiCE • Cox Hazard • Evidently • ReportLab</div>", unsafe_allow_html=True)


Overwriting mlops/deployment/app.py


## Dependency Handling

In [19]:
%%writefile mlops/deployment/requirements.txt
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.43.2
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1

Writing mlops/deployment/requirements.txt


# Hosting

In [20]:
os.makedirs("mlops/hosting", exist_ok=True)

In [21]:
%%writefile mlops/hosting/hosting.py
from huggingface_hub import HfApi, create_repo, login
import os
import time

token = os.getenv("HF_TOKEN")
if token:
    try:
        login(token=token)
    except Exception as login_err:
        print(f"Login notice: {login_err}")

api = HfApi(token=token)
repo_id = "krish21may/Bank-Customer-Churn-4"
repo_type = "space"

# Step 1: Ensure Space exists on Hugging Face
try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists.")
except Exception as e:
    print(f"Space '{repo_id}' not found or error: {e}. Creating new Streamlit Space...")
    try:
        create_repo(
            repo_id=repo_id, 
            repo_type=repo_type, 
            space_sdk="streamlit", 
            private=False, 
            token=token
        )
        print(f"Space '{repo_id}' created successfully.")
    except Exception as create_err:
        print(f"Space creation info: {create_err}")

# Step 2: Upload folder with automatic retry & backoff for rate limits
max_retries = 3
for attempt in range(1, max_retries + 1):
    try:
        print(f"Uploading 'mlops/deployment' to Space '{repo_id}' (Attempt {attempt}/{max_retries})...")
        api.upload_folder(
            folder_path="mlops/deployment",
            repo_id=repo_id,
            repo_type=repo_type,
            path_in_repo="",
        )
        print("Successfully uploaded files to Hugging Face Space!")
        break
    except Exception as upload_err:
        print(f"Upload attempt {attempt} failed: {upload_err}")
        if attempt < max_retries:
            sleep_time = attempt * 5
            print(f"Retrying in {sleep_time} seconds...")
            time.sleep(sleep_time)
        else:
            raise upload_err


Writing mlops/hosting/hosting.py


# Create and Automate MLOps Pipeline with GitHub Action Workflows using CI/CD

## Actions Workflow YAML File

* A YAML file is a simple, human-readable file used to store configuration settings.
* YAML stands for Yet Another Markup Language or YAML Ain't Markup Language (a recursive acronym).
* It uses indentation (spaces) to show structure, like folders inside folders.
* Each line contains a key and a value, making it easy to organize data.
* YAML is often used in automation tools, cloud setups, and app settings.

Here's the YAML file we'd need for our use case.

```
name: MLOps pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch
jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/model_building/prep.py


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server start
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/model_building/train.py

  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: pip install -r mlops/requirements.txt
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: python mlops/hosting/hosting.py

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [22]:
%%writefile mlops/requirements.txt
huggingface_hub==0.32.6
datasets==3.6.0
pandas==2.2.2
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1

Writing mlops/requirements.txt


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.